# Misja 2: szkoła jazdy i robot artysta 🐢

**Cel:** zobaczyć współpracę węzłów, sterować robotem i znaleźć jego kanały komunikacji.
Na koniec wyjaśnisz, jak polecenie wywołane naciśnięciem klawisza dociera do symulatora.
Potrzebujesz uruchomionego kontenera i pulpitu http://localhost:6080 → **Connect**.

Zaczynamy od turtlesim: żółw jest robotem na płaszczyźnie, a kreska pokazuje jego trasę.
Gazebo z robotem kołowym jest dodatkiem na końcu. Wykonuj komórki kolejno.

**Komórka startowa poniżej jest gotowa — nie musisz jej zmieniać.**
`lab` zarządza uruchomieniem symulatora, oczekiwaniem na dane i sprzątaniem.
`node` to Twój węzeł ROS — na nim utworzymy interfejsy komunikacji.
`globals()` przekazuje funkcji pomocniczej stan notebooka, aby przy ponownym
uruchomieniu mogła usunąć stare obiekty.

In [ ]:
from intro_ros import notebook_lab

# Gotowe zaplecze: tworzy node i sprząta poprzednią sesję przy ponownym starcie.
lab = notebook_lab('szkola_jazdy', globals())
node = lab.node
lab.start_turtlesim()

## 1. Pierwszy przejazd

W oknie pulpitu zobaczysz żółwia. Uruchom sterowanie w poniższej komórce, a potem **kliknij okno terminala
sterowania**, aby to ono odbierało naciśnięcia klawiszy. Strzałki ↑/↓ poruszają żółwiem naprzód lub wstecz, a ←/→ go obracają.
Nie są to klawisze `w/a/s/d` używane później przez TurtleBota.
Po puszczeniu klawisza żółw zatrzymuje się po krótkim czasie. **Ctrl+C** zamyka teleop.

In [ ]:
from run_in_term import run_lxterminal

# Uruchom tylko raz. Przy ponownym uruchomieniu najpierw zamknij stary terminal teleop.
run_lxterminal('ros2 run turtlesim turtle_teleop_key')

## 2. Minigra: podpis robota

Masz 90 sekund. Narysuj pierwszą literę swojego imienia, nie dotykając ścian.
Za literę: 1 punkt. Za zamkniętą figurę: kolejny punkt. Za wyjaśnienie partnerowi,
który klawisz zmienia prędkość, a który kierunek: trzeci punkt.

**Przewidź:** czy skręt w lewo zawsze zwiększa współrzędną x? Spróbuj obrócić żółwia o pół obrotu
przed jazdą. „Naprzód” odnosi się do kierunku robota, a x/y do świata.
Jeśli żółw utknie przy brzegu, cofnij. Resetowanie świata poznamy w ćwiczeniu o usługach.

## 3. Detektyw ROS: kto z kim rozmawia?

W terminalu Jupytera lub pulpitu uruchom po kolei:

```bash
ros2 node list
ros2 topic list -t
ros2 topic info /turtle1/cmd_vel --verbose
ros2 topic echo /turtle1/pose --once
ros2 interface show geometry_msgs/msg/Twist
```

**Oczekiwany wynik:** na liście zobaczysz węzeł `/turtlesim` oraz, dopóki działa
sterowanie z klawiatury, węzeł `/teleop_turtle`.
Topic `/turtle1/cmd_vel` ma typ `geometry_msgs/msg/Twist`, a `/turtle1/pose` typ `turtlesim/msg/Pose`.
`echo --once` kończy się po jednej wiadomości; jeśli nic nie przychodzi, przerwij polecenie klawiszami **Ctrl+C**
i sprawdź, czy okno symulatora nadal działa.

**Zadanie:** wskaż publishera i subscribera komend. Zamknij teleop przez Ctrl+C i ponów `topic info`.
Co zniknęło? Poczekaj chwilę na aktualizację grafu.

<details><summary>Podpowiedź</summary>
Teleop publikuje prędkość, turtlesim ją subskrybuje. Żółw publikuje swoje położenie osobnym topikiem.
Węzeł notebooka też pojawi się na liście, choć jeszcze nie wysyła komend.
</details>

In [ ]:
# Ten sam graf można obejrzeć z Pythona. Odkrywanie uczestników trwa chwilę.
lab.spin_for(1.0)
for name, types in node.get_topic_names_and_types():
    if name.startswith('/turtle1/'):
        print(name, types)

## 4. Ruch bez klawiatury

Zamknij teleop, żeby polecenia się nie mieszały. W terminalu wyślij:

```bash
ros2 topic pub --once /turtle1/cmd_vel geometry_msgs/msg/Twist '{linear: {x: 1.0}, angular: {z: 1.0}}'
```

Żółw powinien przejechać krótki łuk. `linear.x` to prędkość naprzód w jednostkach świata na sekundę,
`angular.z` to prędkość kątowa w radianach na sekundę. **To prędkości, nie docelowa pozycja**.
Samo opublikowanie wiadomości nie potwierdza, że ruch nastąpił. Sprawdź to w oknie symulatora lub w danych z topicu `/turtle1/pose`.
Pojedyncza wiadomość nie daje ciągłej kontroli; w ćwiczeniu 3 wyślemy serię komend, a na koniec komendę zerowej prędkości.

**Bonus:** przewidź kształt trasy dla `angular.z = 0.0`, `1.0` i `-1.0`. Sprawdź.

## Dodatek: ten sam pomysł, robot kołowy w Gazebo

To część dla chętnych lub do pokazania przez prowadzącego. Pierwszy start może trwać kilka minut,
a na wolnym komputerze czas w symulacji może płynąć wolniej niż w rzeczywistości.
Ustaw `RUN_GAZEBO = True` tylko wtedy, gdy nie masz już otwartego Gazebo.
Nie uruchamiaj Nav2 podczas ręcznego sterowania.

In [ ]:
RUN_GAZEBO = False
if RUN_GAZEBO:
    run_lxterminal('ros2 launch turtlebot3_gazebo turtlebot3_world.launch.py')
    run_lxterminal('ros2 run turtlebot3_teleop teleop_keyboard')
else:
    print('Dodatek pominięty. Główna misja działa w turtlesim.')

Kliknij terminal sterowania TurtleBotem. Klawisze `w/x` zmieniają prędkość jazdy,
a `a/d` — prędkość obrotu. **Spacja lub klawisz `s` zatrzymuje robota**,
a Ctrl+C kończy sterowanie. Klawisze zmieniają zadaną prędkość,
więc samo puszczenie `w` nie zatrzymuje robota.
Wybierz fragment korytarza, zawróć i zaparkuj w miejscu startu.

Sprawdź `ros2 topic info /cmd_vel --verbose`: w tym warsztatowym obrazie Jazzy
TurtleBot używa **TwistStamped**, z prędkością w `twist.linear.x` i znacznikiem czasu w `header.stamp`.
Nie wysyłaj `Twist` na topic, który wymaga `TwistStamped`.

Podgląd kamery (uruchom tylko, gdy Gazebo jest gotowe):

In [ ]:
if RUN_GAZEBO:
    run_lxterminal('ros2 run rviz2 rviz2 -d /home/ubuntu/turtlebot3_ws/src/jupyter_notebooks/turtlebot_kamera.rviz')

Najpierw uruchom rqt w terminalu kontenera:

```bash
ros2 run rqt_gui rqt_gui
```

Do wykresu w rqt wybierz **Plugins → Visualization → Plot** i pole liczbowe
`/odom/pose/pose/position/x` (samo `position` jest obiektem, nie liczbą).
Graf: **Plugins → Introspection → Node Graph**; odśwież i porównaj z naszym rysunkiem.
Po pokazie zatrzymaj robota (`s`), potem zakończ sterowanie przez Ctrl+C w terminalu teleop.
Zakończ też uruchomienie Gazebo przez Ctrl+C w jego terminalu.
Zamknij też RViz i rqt, zanim przejdziesz do misji 3–5. Dzięki temu ciężka symulacja
nie spowolni ruchów żółwia. Komórka sprzątająca poniżej dotyczy tylko turtlesim.

## Zapisz odkrycie i posprzątaj

Zapisz jednym zdaniem, jakie dane wysyłał lub odbierał Twój węzeł i co działo się dopiero podczas spinowania.
Uruchom poniższą komórkę przed przejściem dalej. Zatrzyma robota, usunie węzeł
i zamknie turtlesim, **jeśli symulator został uruchomiony przez ten notebook**.
Następny notebook w razie potrzeby uruchomi symulator ponownie.
Jeśli przerwiesz wykonywanie kodu, również uruchom tę komórkę. Po restarcie kernela
zamknij ręcznie okno turtlesim, jeśli pozostało otwarte.

In [ ]:
lab.close()

[Dalej: 3. Publisher →](3.%20ROS%20Topic%20-%20Publisher.ipynb)